# 05b — Churn Prediction: XGBoost & LightGBM+XGBoost Ensemble (Optuna)

**Inputs:** `outputs/train.parquet`, `outputs/val.parquet`, `outputs/test.parquet`  
**Outputs:** `outputs/xgb_model.pkl`, `outputs/lgbm_optuna_model.pkl`, `outputs/test_predictions_ensemble.parquet`

| Section | What |
|---|---|
| 1 — Baselines | XGBoost (`scale_pos_weight`) + LightGBM (`class_weight='balanced'`) |
| 2 — XGBoost Optuna | 50-trial TPE search, val ROC-AUC objective |
| 3 — LightGBM Optuna | 50-trial TPE search, val ROC-AUC objective |
| 4 — Comparison | Side-by-side val metrics, both Optuna-tuned models |
| 5 — Ensemble | Soft-vote average of tuned XGBoost + tuned LightGBM |
| 6 — Final | Refit on train+val, test evaluation, ROC curves, save |

**Features (7):** recency · frequency · monetary · aov · purchase_velocity · days_as_customer · unique_products  
**Imbalance:** XGBoost `scale_pos_weight = n_neg/n_pos` · LightGBM `class_weight='balanced'`

In [1]:
import pandas as pd
import numpy as np
import joblib
import optuna
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
TRAIN_PATH      = Path("../outputs/train.parquet")
VAL_PATH        = Path("../outputs/val.parquet")
TEST_PATH       = Path("../outputs/test.parquet")
XGB_PATH        = Path("../outputs/xgb_model.pkl")
LGBM_PATH       = Path("../outputs/lgbm_optuna_model.pkl")
PREDS_PATH      = Path("../outputs/test_predictions_ensemble.parquet")

FEATURE_COLS = [
    # Original RFM (7)
    "recency", "frequency", "monetary", "aov",
    "purchase_velocity", "days_as_customer", "unique_products",
    # Behavioural (9, from notebook 02 Section 2b)
    "avg_inter_purchase_days", "cv_inter_purchase_days",
    "spend_trend", "unique_months_active",
    "cv_order_value", "max_order_value",
    "pct_weekend_orders", "herfindahl_products", "return_rate",
]
TARGET_COL = "churn_label"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    assert p.exists(), f"Run notebook 04 first — {p} not found"

train = pd.read_parquet(TRAIN_PATH)
val   = pd.read_parquet(VAL_PATH)
test  = pd.read_parquet(TEST_PATH)

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()

print(f"train : {X_train.shape}   churn = {y_train.mean()*100:.1f}%")
print(f"val   : {X_val.shape}   churn = {y_val.mean()*100:.1f}%")
print(f"test  : {X_test.shape}   churn = {y_test.mean()*100:.1f}%")
print(f"\nscale_pos_weight = {SCALE_POS_WEIGHT:.2f}  (n_neg / n_pos)")

In [3]:
def eval_metrics(model, X, y, split=""):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    return {
        "split":     split,
        "accuracy":  round(accuracy_score(y, y_pred),                    4),
        "precision": round(precision_score(y, y_pred, zero_division=0),  4),
        "recall":    round(recall_score(y, y_pred),                      4),
        "f1":        round(f1_score(y, y_pred),                          4),
        "roc_auc":   round(roc_auc_score(y, y_prob),                     4),
    }


def ensemble_metrics(prob_a, prob_b, y, split="", threshold=0.5):
    y_prob = (prob_a + prob_b) / 2
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "split":     split,
        "accuracy":  round(accuracy_score(y, y_pred),                    4),
        "precision": round(precision_score(y, y_pred, zero_division=0),  4),
        "recall":    round(recall_score(y, y_pred),                      4),
        "f1":        round(f1_score(y, y_pred),                          4),
        "roc_auc":   round(roc_auc_score(y, y_prob),                     4),
    }


def optuna_history_plot(study, title):
    df = study.trials_dataframe(attrs=("number", "value"))
    df = df.rename(columns={"value": "val_auc"})
    fig = px.line(df, x="number", y="val_auc", title=title,
                  labels={"number": "Trial", "val_auc": "Val ROC-AUC"})
    best_n = study.best_trial.number
    fig.add_vline(x=best_n, line_dash="dash", line_color="red",
                  annotation_text=f"best (#{best_n})", annotation_position="top right")
    fig.show()

---
## Section 1 — Baselines

Default hyperparameters with only imbalance handling applied — establishes the floor
before Optuna tuning in Sections 2 and 3.

- **XGBoost:** `scale_pos_weight = n_neg / n_pos` (~3.5×) — scales minority class in the loss
- **LightGBM:** `class_weight='balanced'` — equivalent upweighting, leaf-wise growth

In [4]:
baseline_xgb = XGBClassifier(
    scale_pos_weight=SCALE_POS_WEIGHT,
    random_state=42, eval_metric="logloss", verbosity=0,
)
baseline_xgb.fit(X_train, y_train)

baseline_lgbm = LGBMClassifier(
    class_weight="balanced",
    random_state=42, verbose=-1,
)
baseline_lgbm.fit(X_train, y_train)

baselines = pd.DataFrame([
    eval_metrics(baseline_xgb,  X_val, y_val, split="XGBoost  (baseline)"),
    eval_metrics(baseline_lgbm, X_val, y_val, split="LightGBM (baseline)"),
]).set_index("split")

print("Baseline val metrics:")
baselines

Baseline val metrics:


,accuracy,precision,recall,f1,roc_auc
split,,,,,
XGBoost (baseline),0.6395,0.3923,0.4679,0.4268,0.6491
LightGBM (baseline),0.5974,0.3830,0.6606,0.4848,0.6580


---
## Section 2 — XGBoost Hyperparameter Tuning (Optuna)

50-trial TPE Bayesian search, val ROC-AUC objective. No cross-validation — val set
evaluated directly to preserve temporal integrity.

| Parameter | Range | Scale |
|---|---|---|
| `max_depth` | 3 – 10 | int |
| `learning_rate` | 0.01 – 0.3 | log-uniform |
| `n_estimators` | 100 – 500 | int |
| `min_child_weight` | 1 – 10 | int |
| `subsample` | 0.6 – 1.0 | uniform |
| `colsample_bytree` | 0.6 – 1.0 | uniform |
| `gamma` | 0 – 5 | uniform |

In [5]:
def xgb_objective(trial):
    params = {
        "max_depth":        trial.suggest_int("max_depth", 3, 10),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators":     trial.suggest_int("n_estimators", 100, 500),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
    }
    mdl = XGBClassifier(
        **params,
        scale_pos_weight=SCALE_POS_WEIGHT,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
    )
    mdl.fit(X_train, y_train)
    return roc_auc_score(y_val, mdl.predict_proba(X_val)[:, 1])


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
best_auc    = study.best_value
print(f"\nBest val ROC-AUC : {best_auc:.4f}  (trial #{study.best_trial.number})")

# Refit best config to get the model object
best_xgb = XGBClassifier(
    **best_params,
    scale_pos_weight=SCALE_POS_WEIGHT,
    random_state=42,
    eval_metric="logloss",
    verbosity=0,
)
best_xgb.fit(X_train, y_train)

  0%|          | 0/50 [00:00<?, ?it/s]


Best val ROC-AUC : 0.7190  (trial #13)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7449716219780114
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import 

In [6]:
print(f"Best val ROC-AUC : {best_auc:.4f}")
print(f"Best params:")
for k, v in best_params.items():
    print(f"  {k:<22} = {v}")

# Optuna trials — top 10 by val AUC
trials_df = study.trials_dataframe(attrs=("number", "value", "params"))
trials_df.columns = [c.replace("params_", "") for c in trials_df.columns]
trials_df = trials_df.rename(columns={"value": "val_auc"}).sort_values("val_auc", ascending=False)
print(f"\nTop 10 Optuna trials:")
print(trials_df.head(10).to_string(index=False))

# Optimisation history plot
fig = px.line(
    trials_df.sort_values("number"),
    x="number", y="val_auc",
    title="Optuna optimisation history — val ROC-AUC per trial",
    labels={"number": "Trial", "val_auc": "Val ROC-AUC"},
)
best_trial_num = study.best_trial.number
fig.add_vline(x=best_trial_num, line_dash="dash", line_color="red",
              annotation_text=f"best (#{best_trial_num})", annotation_position="top right")
fig.show()

Best val ROC-AUC : 0.7190
Best params:
  max_depth              = 3
  learning_rate          = 0.016115655079331635
  n_estimators           = 172
  min_child_weight       = 8
  subsample              = 0.7033351318553821
  colsample_bytree       = 0.7449716219780114
  gamma                  = 1.7139964413315822

Top 10 Optuna trials:
 number  val_auc  colsample_bytree    gamma  learning_rate  max_depth  min_child_weight  n_estimators  subsample
     13 0.719049          0.744972 1.713996       0.016116          3                 8           172   0.703335
     37 0.716578          0.999313 4.348659       0.012289          3                 7           360   0.774009
     45 0.714598          0.927496 4.578068       0.011567          3                 5           306   0.778695
     41 0.714411          0.989255 4.037965       0.025597          3                 6           361   0.773679
     21 0.714158          0.733150 3.917649       0.013668          3                 4           

---
## Section 3 — LightGBM Hyperparameter Tuning (Optuna)

Same 50-trial TPE setup as Section 2, but with LightGBM-specific parameters.
`class_weight='balanced'` is fixed throughout all trials.

| Parameter | Range | Scale |
|---|---|---|
| `num_leaves` | 20 – 200 | int |
| `learning_rate` | 0.01 – 0.3 | log-uniform |
| `n_estimators` | 100 – 500 | int |
| `min_child_samples` | 5 – 50 | int |
| `subsample` | 0.6 – 1.0 | uniform |
| `colsample_bytree` | 0.6 – 1.0 | uniform |
| `reg_alpha` | 0 – 5 | uniform |
| `reg_lambda` | 0 – 5 | uniform |

In [7]:
def lgbm_objective(trial):
    params = {
        "num_leaves":        trial.suggest_int("num_leaves", 20, 200),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.0, 5.0),
    }
    mdl = LGBMClassifier(
        **params,
        class_weight="balanced",
        random_state=42,
        verbose=-1,
    )
    mdl.fit(X_train, y_train)
    return roc_auc_score(y_val, mdl.predict_proba(X_val)[:, 1])


lgbm_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
lgbm_study.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

lgbm_best_params = lgbm_study.best_params
lgbm_best_auc    = lgbm_study.best_value
print(f"\nBest val ROC-AUC : {lgbm_best_auc:.4f}  (trial #{lgbm_study.best_trial.number})")

best_lgbm = LGBMClassifier(
    **lgbm_best_params,
    class_weight="balanced",
    random_state=42,
    verbose=-1,
)
best_lgbm.fit(X_train, y_train)

  0%|          | 0/50 [00:00<?, ?it/s]


Best val ROC-AUC : 0.7036  (trial #44)


,boosting_type,'gbdt'
,num_leaves,38
,max_depth,-1
,learning_rate,0.01183184130740195
,n_estimators,103
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,5


In [8]:
print(f"Best val ROC-AUC : {lgbm_best_auc:.4f}")
print(f"Best params:")
for k, v in lgbm_best_params.items():
    print(f"  {k:<22} = {v}")

trials_lgbm = lgbm_study.trials_dataframe(attrs=("number", "value", "params"))
trials_lgbm.columns = [c.replace("params_", "") for c in trials_lgbm.columns]
trials_lgbm = trials_lgbm.rename(columns={"value": "val_auc"}).sort_values("val_auc", ascending=False)
print(f"\nTop 10 Optuna trials (LightGBM):")
print(trials_lgbm.head(10).to_string(index=False))

optuna_history_plot(lgbm_study, "LightGBM Optuna — optimisation history")

Best val ROC-AUC : 0.7036
Best params:
  num_leaves             = 38
  learning_rate          = 0.01183184130740195
  n_estimators           = 103
  min_child_samples      = 5
  subsample              = 0.6235838618932686
  colsample_bytree       = 0.8037982143359939
  reg_alpha              = 4.5131589665470395
  reg_lambda             = 3.3130831009717334

Top 10 Optuna trials (LightGBM):
 number  val_auc  colsample_bytree  learning_rate  min_child_samples  n_estimators  num_leaves  reg_alpha  reg_lambda  subsample
     44 0.703612          0.803798       0.011832                  5           103          38   4.513159    3.313083   0.623584
     31 0.700227          0.868062       0.010458                  5           236         190   4.968799    3.488218   0.710189
     33 0.699719          0.917997       0.016327                  5           100         144   4.624155    4.040673   0.684555
     38 0.697789          0.601067       0.027699                 17           140        

---
## Section 4 — Optuna-Tuned Model Comparison

Both models tuned on the same val set with 50 Optuna trials. Direct comparison of
val-set metrics shows which model benefits more from Bayesian tuning vs its baseline.

In [9]:
lgbm_val_metrics = eval_metrics(best_lgbm, X_val, y_val, split="LightGBM (Optuna)")
xgb_val_metrics  = eval_metrics(best_xgb,  X_val, y_val, split="XGBoost  (Optuna)")

comparison = pd.DataFrame([
    lgbm_val_metrics,
    xgb_val_metrics,
]).set_index("split")

print("Val set comparison — both Optuna-tuned:")
comparison

Val set comparison — both Optuna-tuned:


,accuracy,precision,recall,f1,roc_auc
split,,,,,
LightGBM (Optuna),0.5000,0.3483,0.8532,0.4947,0.7036
XGBoost (Optuna),0.5474,0.3755,0.8716,0.5249,0.7190


---
## Section 5 — Soft-Voting Ensemble

Average predicted probabilities from the two Optuna-tuned models (soft voting).

```
ensemble_prob = (lgbm_prob + xgb_prob) / 2
```

In [10]:
lgbm_prob_val = best_lgbm.predict_proba(X_val)[:, 1]
xgb_prob_val  = best_xgb.predict_proba(X_val)[:, 1]

ens_val_metrics = ensemble_metrics(lgbm_prob_val, xgb_prob_val, y_val, split="Ensemble")

three_way = pd.DataFrame([
    {**lgbm_val_metrics, "split": "LightGBM (Optuna)"},
    {**xgb_val_metrics,  "split": "XGBoost  (Optuna)"},
    {**ens_val_metrics,  "split": "Ensemble"},
]).set_index("split")

print("Val set — three-way comparison:")
three_way

Val set — three-way comparison:


,accuracy,precision,recall,f1,roc_auc
split,,,,,
LightGBM (Optuna),0.5000,0.3483,0.8532,0.4947,0.7036
XGBoost (Optuna),0.5474,0.3755,0.8716,0.5249,0.7190
Ensemble,0.5421,0.3725,0.8716,0.5220,0.7161


---
## Section 6 — Test Set Evaluation & Outputs

Refit both Optuna-tuned models on **train+val combined** before final test evaluation.
The test set has not been touched until this point.

In [11]:
train_val = pd.concat([train, val], ignore_index=True)
X_tv = train_val[FEATURE_COLS]
y_tv = train_val[TARGET_COL]

scale_pos_weight_tv = (y_tv == 0).sum() / (y_tv == 1).sum()

final_xgb = XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight_tv,
    random_state=42, eval_metric="logloss", verbosity=0,
)
final_xgb.fit(X_tv, y_tv)

final_lgbm = LGBMClassifier(
    **lgbm_best_params,
    class_weight="balanced",
    random_state=42, verbose=-1,
)
final_lgbm.fit(X_tv, y_tv)

print(f"Final XGBoost  trained on {len(X_tv):,} samples (train+val combined)")
print(f"Final LightGBM trained on {len(X_tv):,} samples (train+val combined)")

Final XGBoost  trained on 2,542 samples (train+val combined)
Final LightGBM trained on 2,542 samples (train+val combined)


In [12]:
lgbm_prob_test = final_lgbm.predict_proba(X_test)[:, 1]
xgb_prob_test  = final_xgb.predict_proba(X_test)[:, 1]

lgbm_test_metrics = eval_metrics(final_lgbm, X_test, y_test, split="LightGBM (Optuna)")
xgb_test_metrics  = eval_metrics(final_xgb,  X_test, y_test, split="XGBoost  (Optuna)")
ens_test_metrics  = ensemble_metrics(lgbm_prob_test, xgb_prob_test, y_test, split="Ensemble")

test_comparison = pd.DataFrame([
    lgbm_test_metrics,
    xgb_test_metrics,
    ens_test_metrics,
]).set_index("split")

print("Test set — final three-way comparison:")
test_comparison

Test set — final three-way comparison:


,accuracy,precision,recall,f1,roc_auc
split,,,,,
LightGBM (Optuna),0.6314,0.3714,0.4815,0.4194,0.6200
XGBoost (Optuna),0.6348,0.3952,0.6049,0.4780,0.6447
Ensemble,0.6246,0.3717,0.5185,0.4330,0.6369


In [13]:
ens_prob_test = (lgbm_prob_test + xgb_prob_test) / 2

fig = go.Figure()

for name, probs, color in [
    ("LightGBM",  lgbm_prob_test, "#2196F3"),
    ("XGBoost",   xgb_prob_test,  "#FF9800"),
    ("Ensemble",  ens_prob_test,  "#4CAF50"),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode="lines",
        name=f"{name} (AUC = {auc:.3f})",
        line=dict(color=color, width=2),
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    name="Random baseline",
    line=dict(color="grey", dash="dash"),
))
fig.update_layout(
    title="ROC Curves — test set (LightGBM vs XGBoost vs Ensemble)",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    legend=dict(x=0.55, y=0.1),
)
fig.show()

In [14]:
importance_df = pd.DataFrame({
    "feature":    FEATURE_COLS,
    "importance": final_xgb.feature_importances_,
}).sort_values("importance", ascending=True)

fig = px.bar(
    importance_df, x="importance", y="feature",
    orientation="h",
    title="XGBoost feature importance (gain)",
    labels={"importance": "Importance", "feature": "Feature"},
    color="importance",
    color_continuous_scale="Oranges",
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

In [15]:
joblib.dump(final_xgb,  XGB_PATH)
joblib.dump(final_lgbm, LGBM_PATH)

ens_prob_test = (lgbm_prob_test + xgb_prob_test) / 2
preds_df = test[["customer_id", TARGET_COL]].copy()
preds_df["lgbm_prob"]     = lgbm_prob_test
preds_df["xgb_prob"]      = xgb_prob_test
preds_df["ensemble_prob"] = ens_prob_test
preds_df.to_parquet(PREDS_PATH, engine="pyarrow", index=False)

assert XGB_PATH.exists() and LGBM_PATH.exists()
chk = pd.read_parquet(PREDS_PATH)
assert list(chk.columns) == ["customer_id", "churn_label", "lgbm_prob", "xgb_prob", "ensemble_prob"]
assert chk[["lgbm_prob", "xgb_prob", "ensemble_prob"]].apply(lambda c: c.between(0, 1).all()).all()

print(f"XGBoost model  saved → {XGB_PATH.resolve()}")
print(f"LightGBM model saved → {LGBM_PATH.resolve()}")
print(f"Ensemble preds saved → {PREDS_PATH.resolve()}  ({len(chk):,} rows)")
print(f"\nTest set final metrics:")
print(test_comparison.to_string())

XGBoost model  saved → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\xgb_model.pkl
LightGBM model saved → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\lgbm_optuna_model.pkl
Ensemble preds saved → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\test_predictions_ensemble.parquet  (293 rows)

Test set final metrics:
                   accuracy  precision  recall      f1  roc_auc
split                                                          
LightGBM (Optuna)    0.6314     0.3714  0.4815  0.4194   0.6200
XGBoost  (Optuna)    0.6348     0.3952  0.6049  0.4780   0.6447
Ensemble             0.6246     0.3717  0.5185  0.4330   0.6369
